In [1]:
import numpy as np
import cobra
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pickle
import sys
sys.path.append('../../')
from helper_function import *
import gc
import pandas as pd
from scipy.sparse import load_npz

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 16

In [2]:
with open('../../FBAsolution/M9_sufficient_400_40_proteome_weakbg.pkl', 'rb') as pickle_file:
    data2 = pickle.load(pickle_file)
    all_total_nutrient_count2 = data2['all_total_nutrient_count']
    all_total_intake2 = data2['all_total_intake']
    all_growth_rate2 = data2['all_growth_rate']
    all_random_medium2 = data2['all_random_medium']
    all_met_flux2 = data2['all_met_flux']
    del data2

with open('../../FBAsolution/M9_sufficient_20_40_proteome_weakbg.pkl', 'rb') as pickle_file:
    data = pickle.load(pickle_file)
    all_total_nutrient_count = data['all_total_nutrient_count']
    all_total_intake = data['all_total_intake']
    all_growth_rate = data['all_growth_rate']
    all_random_medium = data['all_random_medium']
    all_met_flux = data['all_met_flux']
    del data

with open('../../Coarse_graining_result/clustered_nodes_c20_s44_pq1_relabeled.pkl', 'rb') as f:
    cluster_nodes = pickle.load(f)

In [ ]:
def node_proteome_cost(met_flux, node_id, cluster_nodes):
        
    grouped_rec = [rec for rec in cluster_nodes[node_id] if 'EX_' not in rec]
    #grouped_flux = {rec: value for rec, value in met_flux if rec+'_fwd' in grouped_rec  or rec+'_bwd' in grouped_rec}

    grouped_flux = {rec: value for rec, value in met_flux if (rec+'_fwd' in grouped_rec and value > 0) or (rec+'_bwd' in grouped_rec and value < 0)}

    node_proteome_cost = 0
    for rec, flux in grouped_flux.items():
        proteome_cost = abs(flux)
        node_proteome_cost += proteome_cost

    return node_proteome_cost

def graph_node_proteome_cost(met_flux, cluster_nodes):
    graph_proteome_cost = {}

    for node_id in range(20):
        temp = node_proteome_cost(met_flux, node_id, cluster_nodes)
        graph_proteome_cost[node_id] = temp

    return graph_proteome_cost

def graph_node_proteome_allocation(met_flux, cluster_nodes):
    cost = graph_node_proteome_cost(met_flux, cluster_nodes)
    E_tmp = list(cost.values())
    try:
        with np.errstate(divide='raise', invalid='raise'):
            E_arr = np.asarray(E_tmp, dtype=float)
            E = E_arr / E_arr.sum()
            return E
    except FloatingPointError:
            return []

In [ ]:
def graph_to_row_normalized_matrix(G):
    # Get adjacency matrix as float numpy array
    A = nx.adjacency_matrix(G).toarray().astype(float)

    # Compute row sums
    row_sums = A.sum(axis=1, keepdims=True)

    # Avoid division by zero (rows with sum 0 stay as 0 distribution)
    row_sums[row_sums == 0] = 1

    # Row normalization
    P = A / row_sums

    return P

import numpy as np

def gini_coefficient(x):
    x = np.array(x, dtype=float)
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    cum = np.cumsum(x)
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n

def sector_flux_distribution_vector(met_flux, cluster_nodes):

    n_sectors = len(cluster_nodes)
    sector_flux = []

    for rxn_idx, s in cluster_nodes.items():
        grouped_rec = [rec for rec in cluster_nodes[rxn_idx] if 'EX_' not in rec]
        grouped_flux = {rec: abs(value) for rec, value in met_flux if (rec+'_fwd' in grouped_rec and value > 0) or (rec+'_bwd' in grouped_rec and value < 0)}
        sector_flux.append(list(grouped_flux.values()))
    


    total_gini = []
    total_entropy = [] 
    for s in range(n_sectors):
        flux_arr = np.array(sector_flux[s], dtype=float)
        if flux_arr.size == 0 or np.all(flux_arr == 0):
            gini = 0.0
            entropy = 0.0
        else:
            p = flux_arr / flux_arr.sum()
            p_nonzero = p[p > 0]
            entropy = -np.sum(p_nonzero * np.log(p_nonzero))
            gini = gini_coefficient(p)
        total_gini.append(gini)
        total_entropy.append(entropy)

    return np.array(total_gini, dtype=float), np.array(total_entropy, dtype=float)


In [ ]:
df = []
for i in range(40):
    gr = []
    comparison = []
    for j, (met_flux, met_flux2) in enumerate(zip(all_met_flux[i], all_met_flux2[i])):
        try:
            E1 = graph_node_proteome_allocation(met_flux, cluster_nodes)
            E2 = graph_node_proteome_allocation(met_flux2, cluster_nodes)

            gini_1, entropy_1 = sector_flux_distribution_vector(met_flux, cluster_nodes)
            gini_2, entropy_2 = sector_flux_distribution_vector(met_flux2, cluster_nodes)

            df.append((20, j, i, gini_1, entropy_1, E1, all_growth_rate[i][j], all_total_nutrient_count[i][j]))
            df.append((400, j, i, gini_2, entropy_2, E2, all_growth_rate2[i][j], all_total_nutrient_count2[i][j]))

        except Exception as e:
            print(e)
            continue

In [ ]:
df = pd.DataFrame(df, columns=['M', 'Species_index', 'Env_index', 'Gini', 'Entropy', 'E', 'lambda', 'N'])
df.to_csv('flux_flow_graph_comparision_across_entropy_ei_new.csv', index=False)